# Notebook B: Combining /accounts, /groups, and /users

**Purpose:** Pull records from three related FOLIO endpoints and join them into a
single analysis-ready DataFrame. This is the kind of cross-record combination the
Stripes UI doesn't do natively.

**Assumed relationships** (confirm against your instance — I have not verified these
field names against current Sunflower schema, so treat them as a starting hypothesis
to check, not a fact):
- `/accounts` records reference a user via a `userId` field.
- `/users` records reference a patron group via a `patronGroup` field (a group's `id`).
- `/groups` records have an `id` and a human-readable `group` name.

If any of those field names are wrong for your instance, the fix is just changing the
column name in the merge step below — the overall approach stays the same.

**How to use this notebook:** Each section has a markdown cell explaining the step,
followed by a code cell. `# TODO (FOLIO-specific)` marks spots to confirm/adjust
against your actual schema.


## 1. Environment setup


In [127]:
# !pip install pandas requests

import pandas as pd
import requests

pd.set_option('display.max_columns', None)


## 2. Login
### Note: This script references patron data, so the login credentials must be able to access user accounts in FOLIO. 


In [128]:

%run folio_auth.ipynb


Login succeeded. Token retrieved.


## 3. A small helper for paginated GET requests

FOLIO endpoints typically page results via `limit`/`offset` query params and return a
JSON object with a records array plus a total count. This helper loops until it has
everything. **Confirm the pagination param names and the response envelope key names
against your instance** — I'm using common FOLIO conventions here, but I have not
verified them against current docs.


In [136]:
def fetch_all_records(endpoint, records_key, limit=1000):
    """
    Fetch all records from a paginated FOLIO endpoint.

    endpoint: path like "/accounts", "/groups", "/users"
    records_key: the JSON key holding the list of records, e.g. "accounts", "usergroups", "users"
    """
    all_records = []
    offset = 0
    base_url = OKAPI_URL.copy()
    headers = HEADERS.copy()


    while True:
        response = requests.get(
            f"{base_url}{endpoint}",
            headers=headers,
            params={"limit": limit, "offset": offset},
        )
        response.raise_for_status()  # fail loudly and clearly if something's wrong
        payload = response.json()

        batch = payload.get(records_key, [])
        all_records.extend(batch)

        if len(batch) < limit:
            break  # last page
        offset += limit

    return all_records


## 4. Pull data from each endpoint

TODO (FOLIO-specific): confirm the `records_key` for each — FOLIO's convention is
usually the plural of the resource, but it varies (e.g. `/groups` often returns
`"usergroups"` rather than `"groups"` — **check this**, I'm not certain of the exact
key for your instance).


In [137]:
# redefine authenticated fetch that uses existing session, OKAPI_URL, HEADERS, and token
def fetch_all_records(endpoint, records_key, limit=1000):
    all_records = []
    offset = 0
    base_url = OKAPI_URL
    headers = HEADERS.copy() if 'HEADERS' in globals() else {"X-Okapi-Tenant": TENANT, "Content-Type": "application/json"}
    if 'token' in globals():
        headers["Authorization"] = f"Bearer {token}"

    while True:
        response = session.get(f"{base_url}{endpoint}", headers=headers, params={"limit": limit, "offset": offset})
        response.raise_for_status()
        payload = response.json()

        batch = payload.get(records_key, [])
        all_records.extend(batch)

        if len(batch) < limit:
            break
        offset += limit

    return all_records

accounts_raw = fetch_all_records("/accounts", records_key="accounts")
groups_raw   = fetch_all_records("/groups",   records_key="usergroups")  
users_raw    = fetch_all_records("/users",    records_key="users")

print(f"accounts: {len(accounts_raw)}")
print(f"groups:   {len(groups_raw)}")
print(f"users:    {len(users_raw)}")


accounts: 1323
groups:   13
users:    1535


In [138]:
accounts_df = pd.DataFrame(accounts_raw)
groups_df   = pd.DataFrame(groups_raw)
users_df    = pd.DataFrame(users_raw)

accounts_df.head()


,amount,remaining,status,paymentStatus,feeFineType,feeFineOwner,callNumber,metadata,userId,feeFineId,ownerId,id,contributors,title,barcode,materialType,location,dueDate,loanId,itemId,materialTypeId,loanPolicyId,overdueFinePolicyId,lostItemFeePolicyId,holdingsRecordId,instanceId,returnedDate
0,5.0,0.0,{'name': 'Closed'},{'name': 'Transferred fully'},Damaged item,Ipswich Campus,,{'createdDate': '2022-12-01T14:02:25.906+00:00...,af74f17b-24a3-42be-ae2a-b4137dd73684,5d384949-3a3c-40d2-b93e-174a3a6c459c,413384df-35a7-4dac-aeaa-cc7f178b3927,4157cfa3-a007-4301-b078-0dd9cee066eb,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,10.0,0.0,{'name': 'Closed'},{'name': 'Cancelled item returned'},Lost item processing fee,CMP Contoocook Campus,M1366 .E433 C6 1972,{'createdDate': '2023-10-05T17:46:07.147+00:00...,f7ea94c2-b17e-4b4c-bb9e-91538b11987e,c7dede15-aa48-45ed-860b-f996540180e0,476a7fb1-f289-449a-b300-15f8221b3d5e,0aed4f75-bebb-4059-af09-b006ae42307f,"[{'name': 'Ellis, Don'}]",Connection.,32260002413492,Sound recording,CMP Media Collection,2023-11-05T03:59:59.000+00:00,9c020464-72bd-4930-840c-457c78c97f37,ae46cd7b-ab16-4595-a20c-3d8f341a16c5,991e9cad-6318-47f5-9726-dce7053e6a6b,ce787fcd-2ba9-4b18-b232-e61b4e93d5db,612518cf-8597-4685-8d53-b9a1e2053a94,cb18f7aa-8af4-4b6d-965c-ae61fe91626c,NaN,NaN,NaN
2,10.0,0.0,{'name': 'Closed'},{'name': 'Paid fully'},Damaged item,Birmingham Campus,NaN,{'createdDate': '2022-05-10T14:51:33.262+00:00...,693b40a5-a9f5-4d00-bea5-bc144dbfa6d8,37d12bf2-d337-4258-8bd3-ad47c05e0b5d,b4f7129d-6658-4e74-bf26-07d7dc1c551e,d62f6c95-8727-419c-a302-b81669ed9345,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2.0,0.0,{'name': 'Closed'},{'name': 'Waived fully'},Replacement card,Birmingham Campus,,{'createdDate': '2023-02-07T17:55:28.533+00:00...,693b40a5-a9f5-4d00-bea5-bc144dbfa6d8,993593ed-db1f-4cff-9271-69a5bcc7aab2,b4f7129d-6658-4e74-bf26-07d7dc1c551e,0f2bbfe5-119b-4228-91e9-e13cf62c3d19,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,15.0,0.0,{'name': 'Closed'},{'name': 'Waived fully'},Damaged item,Birmingham Campus,NaN,{'createdDate': '2022-05-10T11:32:46.276+00:00...,693b40a5-a9f5-4d00-bea5-bc144dbfa6d8,37d12bf2-d337-4258-8bd3-ad47c05e0b5d,b4f7129d-6658-4e74-bf26-07d7dc1c551e,25d68999-2e6f-4e79-8c9d-9be50cfa8e1a,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 5. Inspect join keys before merging

This is the step worth slowing down for. Before merging, check:
- Do the key columns actually exist under the names you expect?
- Are the data types consistent between the two sides of each join (e.g. both strings)?
- Any leading/trailing whitespace or case differences?


In [139]:
print(accounts_df.columns.tolist())
print(users_df.columns.tolist())
print(groups_df.columns.tolist())


['amount', 'remaining', 'status', 'paymentStatus', 'feeFineType', 'feeFineOwner', 'callNumber', 'metadata', 'userId', 'feeFineId', 'ownerId', 'id', 'contributors', 'title', 'barcode', 'materialType', 'location', 'dueDate', 'loanId', 'itemId', 'materialTypeId', 'loanPolicyId', 'overdueFinePolicyId', 'lostItemFeePolicyId', 'holdingsRecordId', 'instanceId', 'returnedDate']
['username', 'id', 'active', 'patronGroup', 'departments', 'proxyFor', 'personal', 'createdDate', 'updatedDate', 'metadata', 'preferredEmailCommunication', 'externalSystemId', 'barcode', 'type', 'customFields', 'expirationDate', 'tags', 'enrollmentDate']
['group', 'desc', 'id', 'metadata', 'expirationOffsetInDays']


In [133]:
# Spot-check types of the columns you intend to join on
# TODO (FOLIO-specific): adjust column names if yours differ
print(accounts_df['userId'].dtype)
print(users_df['id'].dtype)
print(users_df['patronGroup'].dtype)
print(groups_df['id'].dtype)


str
str
str
str


## 6. Merge

Two joins: accounts → users, then that result → groups.

Starting with `how='left'` keeps every account row even if a match isn't found, so you
can see what didn't match rather than silently losing rows.


In [134]:
# Step 1: accounts + users
accounts_users = accounts_df.merge(
    users_df,
    left_on='userId',
    right_on='id',
    how='left',
    suffixes=('_account', '_user'),
)

# Step 2: + groups
full_df = accounts_users.merge(
    groups_df,
    left_on='patronGroup',
    right_on='id',
    how='left',
    suffixes=('', '_group'),
)

full_df.head()


,amount,remaining,status,paymentStatus,feeFineType,feeFineOwner,callNumber,metadata_account,userId,feeFineId,ownerId,id_account,contributors,title,barcode_account,materialType,location,dueDate,loanId,itemId,materialTypeId,loanPolicyId,overdueFinePolicyId,lostItemFeePolicyId,holdingsRecordId,instanceId,returnedDate,username,id_user,active,patronGroup,departments,proxyFor,personal,createdDate,updatedDate,metadata_user,preferredEmailCommunication,externalSystemId,barcode_user,type,customFields,expirationDate,tags,enrollmentDate,group,desc,id,metadata,expirationOffsetInDays
0,5.0,0.0,{'name': 'Closed'},{'name': 'Transferred fully'},Damaged item,Ipswich Campus,,{'createdDate': '2022-12-01T14:02:25.906+00:00...,af74f17b-24a3-42be-ae2a-b4137dd73684,5d384949-3a3c-40d2-b93e-174a3a6c459c,413384df-35a7-4dac-aeaa-cc7f178b3927,4157cfa3-a007-4301-b078-0dd9cee066eb,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,halexander,af74f17b-24a3-42be-ae2a-b4137dd73684,False,cf99d0df-0c3b-42f6-9037-9e2c28bb771d,[],[],"{'lastName': 'Alexander', 'firstName': 'Henry'...",2026-07-13T20:12:56.087+00:00,2026-07-13T20:12:56.087+00:00,{'createdDate': '2022-04-12T22:29:00.510+00:00...,[],halexander,22874000506277,object,"{'campusAffiliation': 'opt_3', 'graduationYear...",2026-07-30T23:59:59.000+00:00,{'tagList': []},2021-09-01T00:00:00.000+00:00,Student,Matriculating students (including undergraduat...,cf99d0df-0c3b-42f6-9037-9e2c28bb771d,{'createdDate': '2022-04-11T15:58:52.987+00:00...,NaN
1,10.0,0.0,{'name': 'Closed'},{'name': 'Cancelled item returned'},Lost item processing fee,CMP Contoocook Campus,M1366 .E433 C6 1972,{'createdDate': '2023-10-05T17:46:07.147+00:00...,f7ea94c2-b17e-4b4c-bb9e-91538b11987e,c7dede15-aa48-45ed-860b-f996540180e0,476a7fb1-f289-449a-b300-15f8221b3d5e,0aed4f75-bebb-4059-af09-b006ae42307f,"[{'name': 'Ellis, Don'}]",Connection.,32260002413492,Sound recording,CMP Media Collection,2023-11-05T03:59:59.000+00:00,9c020464-72bd-4930-840c-457c78c97f37,ae46cd7b-ab16-4595-a20c-3d8f341a16c5,991e9cad-6318-47f5-9726-dce7053e6a6b,ce787fcd-2ba9-4b18-b232-e61b4e93d5db,612518cf-8597-4685-8d53-b9a1e2053a94,cb18f7aa-8af4-4b6d-965c-ae61fe91626c,NaN,NaN,NaN,fbennett2,f7ea94c2-b17e-4b4c-bb9e-91538b11987e,True,cf99d0df-0c3b-42f6-9037-9e2c28bb771d,[],[],"{'lastName': 'Bennett', 'firstName': 'Fiona', ...",2023-09-06T13:28:26.579+00:00,2023-09-06T13:28:26.579+00:00,{'createdDate': '2022-04-12T22:31:35.753+00:00...,[],fbennett2,22874000506244,object,"{'campusAffiliation': 'opt_2', 'graduationYear...",NaN,NaN,2021-09-01T00:00:00.000+00:00,Student,Matriculating students (including undergraduat...,cf99d0df-0c3b-42f6-9037-9e2c28bb771d,{'createdDate': '2022-04-11T15:58:52.987+00:00...,NaN
2,10.0,0.0,{'name': 'Closed'},{'name': 'Paid fully'},Damaged item,Birmingham Campus,NaN,{'createdDate': '2022-05-10T14:51:33.262+00:00...,693b40a5-a9f5-4d00-bea5-bc144dbfa6d8,37d12bf2-d337-4258-8bd3-ad47c05e0b5d,b4f7129d-6658-4e74-bf26-07d7dc1c551e,d62f6c95-8727-419c-a302-b81669ed9345,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,mdriscoll,693b40a5-a9f5-4d00-bea5-bc144dbfa6d8,True,dbd849b5-ee14-4fc8-9df8-77a29e5db9d1,[93697efb-4e44-4683-aa68-3c5e06d97195],[],"{'lastName': 'Driscoll', 'firstName': 'Molly',...",2024-03-28T13:15:01.626+00:00,2024-03-28T13:15:01.626+00:00,{'createdDate': '2022-04-11T16:03:14.854+00:00...,[],mdriscoll@ebsco.com,22874000505504,NaN,"{'studentId': '22874000505504', 'campusAffilia...",NaN,NaN,2022-04-12T04:00:00.000+00:00,Staff,Campus staff,dbd849b5-ee14-4fc8-9df8-77a29e5db9d1,{'createdDate': '2022-04-11T15:58:52.892+00:00...,NaN
3,2.0,0.0,{'name': 'Closed'},{'name': 'Waived fully'},Replacement card,Birmingham Campus,,{'createdDate': '2023-02-07T17:55:28.533+00:00...,693b40a5-a9f5-4d00-bea5-bc144dbfa6d8,993593ed-db1f-4cff-9271-69a5bcc7aab2,b4f7129d-6658-4e74-bf26-07d7dc1c551e,0f2bbfe5-119b-4228-91e9-e13cf62c3d19,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,mdriscoll,693b40a5-a9f5-4d00-bea5-bc144dbfa6d8,True,dbd849b5-e

## 7. Validate the merge

Common beginner pitfall: a one-to-many relationship silently multiplying rows, or a
type mismatch causing everything to come back unmatched. Check both.


In [135]:
print("Original accounts rows:", len(accounts_df))
print("After merging with users:  ", len(accounts_users))
print("After merging with groups: ", len(full_df))

# Rows where the user or group match failed — worth investigating, not ignoring
unmatched_users = full_df[full_df['id_user'].isnull()] if 'id_user' in full_df.columns else pd.DataFrame()
print("Accounts with no matching user:", len(unmatched_users))


Original accounts rows: 1323
After merging with users:   1323
After merging with groups:  1323
Accounts with no matching user: 12


## 8. Analyze the combined dataset

Now that accounts, users, and groups are joined, you can ask questions that span all
three — e.g. total fee/fine amounts by patron group. Adjust field names to match your
actual `/accounts` schema (commonly something like `amount` or `remaining`).


In [82]:
# TODO (FOLIO-specific): confirm the actual fee/fine amount field name
summary = full_df.groupby('group')['remaining'].sum().sort_values(ascending=False)
print(summary)


group
Student                 3449.99
Staff                   2647.00
Faculty                 2564.00
Community                730.00
zEBSCO Support Group     674.00
MC Student               180.00
Book club                 12.00
Interlibrary loan         10.00
Reading Room Users         2.00
Name: remaining, dtype: float64
